# 02 - Transforms Usage

This notebook is a practical guide to Nebula Space Toolkit's `nstk.transforms` module.

It walks through:
- geodetic, ECEF, ENU, and AER conversions with the unified scalar/array APIs
- coarse ECI/ECEF/geodetic transforms (position and velocity)
- calling unified transform APIs from your own `@numba.njit` functions
- timed Orekit-backed frame transforms via `transform(...)`
- core constants and the unified calling pattern used throughout the module


## Conventions And Naming

- Angular inputs are radians unless a docstring explicitly says otherwise.
- Distances are meters, velocities are m/s, and accelerations are m/s^2.
- The transform module exposes **one public function name per transform family**.
- For position-like transforms, the unified calling pattern is:
  - scalar components: `func(a, b, c, ...)`
  - split 1D arrays: `func(a_vec, b_vec, c_vec, ...)`
  - stacked arrays: `func(stack_nx3, ...)`
- Return values follow the same rule:
  - scalar input returns a 3-tuple of scalars
  - array input returns a 3-tuple of 1D arrays
- Degree output is requested with `degrees=True` when a transform supports it.
- All public transform APIs shown here except `transform(...)` are callable from `@numba.njit` functions.


In [1]:
# Ensure local package import when running from this examples/ folder.
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_repo_root = _cwd if (_cwd / "nstk").is_dir() else _cwd.parent
if (_repo_root / "nstk").is_dir() and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))


In [2]:
from time import perf_counter

import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.utils import iers
from numba import njit

import nstk.transforms as tf
from nstk.time_utils import astropy_time_to_orekit_date

np.set_printoptions(precision=6, suppress=True)

# Avoid long network waits when UT1/IERS tables are requested offline.
iers.conf.auto_download = False
iers.conf.auto_max_age = None


## Public API Map

This cell lists every callable exported by `nstk.transforms`.
Notice that the coordinate conversion APIs are shape-aware under a single public name.


In [3]:
callable_exports = sorted([name for name in tf.__all__ if callable(getattr(tf, name))])

categories = {
    "Geodetic/ECEF": [n for n in callable_exports if "geodetic" in n or "ecef" in n and "coarse" not in n and n != "transform"],
    "ENU": [n for n in callable_exports if "enu" in n and "coarse" not in n],
    "AER": [n for n in callable_exports if "aer" in n],
    "Coarse ECI/ECEF": [n for n in callable_exports if n.startswith("coarse_")],
    "Timed Frame": [n for n in callable_exports if n == "transform"],
}

print(f"Total callable exports: {len(callable_exports)}")
for k, v in categories.items():
    unique = sorted(set(v))
    print(f"\n{k} ({len(unique)}):")
    print(", ".join(unique))


Total callable exports: 22

Geodetic/ECEF (14):
aer2ecef, aer2geodetic, coarse_eci2geodetic, ecef2aer, ecef2enu, ecef2enu_delta, ecef2geodetic, enu2ecef, enu2ecef_delta, enu2geodetic, enu_basis_from_ecef_xyz, geodetic2aer, geodetic2ecef, geodetic2enu

ENU (10):
aer2enu, ecef2enu, ecef2enu_delta, enu2aer, enu2ecef, enu2ecef_delta, enu2geodetic, enu_basis_from_ecef_xyz, enu_basis_from_latlon, geodetic2enu

AER (6):
aer2ecef, aer2enu, aer2geodetic, ecef2aer, enu2aer, geodetic2aer

Coarse ECI/ECEF (5):
coarse_ecef2eci_pos, coarse_ecef2eci_pos_vel, coarse_eci2ecef_pos, coarse_eci2ecef_pos_vel, coarse_eci2geodetic

Timed Frame (1):
transform


## Reference Scenario

Create a single observer and a small set of target points that we reuse in all transform families.


In [4]:
def wrap_pi(rad):
    return (rad + np.pi) % (2.0 * np.pi) - np.pi


obs_lat_deg = 34.0
obs_lon_deg = -118.0
obs_h_m = 250.0

obs_lat = np.deg2rad(obs_lat_deg)
obs_lon = np.deg2rad(obs_lon_deg)

# Targets near the observer.
targets_deg_m = np.array(
    [
        [34.0500, -117.8500, 700.0],
        [34.0200, -118.0200, 1200.0],
        [33.9800, -117.9000, 50.0],
        [34.1200, -118.1500, 3000.0],
    ],
    dtype=np.float64,
)

lat_vec = np.deg2rad(targets_deg_m[:, 0])
lon_vec = np.deg2rad(targets_deg_m[:, 1])
h_vec = targets_deg_m[:, 2]

print("Observer [deg,deg,m]:", obs_lat_deg, obs_lon_deg, obs_h_m)
print("Targets shape:", targets_deg_m.shape)


Observer [deg,deg,m]: 34.0 -118.0 250.0
Targets shape: (4, 3)


## 1) Geodetic <-> ECEF (Scalar)


In [5]:
lat = lat_vec[0]
lon = lon_vec[0]
h_m = float(h_vec[0])

x_m, y_m, z_m = tf.geodetic2ecef(lat, lon, h_m)
lat_back, lon_back, h_back = tf.ecef2geodetic(x_m, y_m, z_m)
lat_back_deg, lon_back_deg, h_back_deg = tf.ecef2geodetic(x_m, y_m, z_m, degrees=True)

print("ECEF [m]:", x_m, y_m, z_m)
print("roundtrip lat err [rad]:", float(lat_back - lat))
print("roundtrip lon err [rad]:", float(wrap_pi(lon_back - lon)))
print("roundtrip h err [m]:", float(h_back - h_m))
print("roundtrip [deg,deg,m]:", lat_back_deg, lon_back_deg, h_back_deg)


ECEF [m]: -2471611.3017184916 -4677928.37587704 3551435.1112874625
roundtrip lat err [rad]: 5.551115123125783e-16
roundtrip lon err [rad]: 0.0
roundtrip h err [m]: 1.862645149230957e-09
roundtrip [deg,deg,m]: 34.05000000000003 -117.85 700.0000000018626


## 2) Geodetic <-> ECEF (Unified Array Interface)

Demonstrates the unified vector interface in this family:
- `geodetic2ecef(lat_vec, lon_vec, h_vec)` for split arrays
- `geodetic2ecef(lla_rad_m)` for an `(N, 3)` array
- `ecef2geodetic(x_vec, y_vec, z_vec)` for split arrays
- `ecef2geodetic(r_ecef_m)` for an `(N, 3)` array
- `ecef2geodetic(..., degrees=True)` for degree output


In [6]:
x_arr, y_arr, z_arr = tf.geodetic2ecef(lat_vec, lon_vec, h_vec)
r_ecef_from_llh = np.column_stack((x_arr, y_arr, z_arr))

lla_rad_m = np.column_stack((lat_vec, lon_vec, h_vec))
r_ecef_from_lla = np.column_stack(tf.geodetic2ecef(lla_rad_m))

lat_xyz, lon_xyz, h_xyz = tf.ecef2geodetic(x_arr, y_arr, z_arr)
lat_ecef, lon_ecef, h_ecef = tf.ecef2geodetic(r_ecef_from_llh)
lat_deg, lon_deg, h_deg = tf.ecef2geodetic(r_ecef_from_llh, degrees=True)

print("geodetic2ecef split-array vs Nx3 allclose:", bool(np.allclose(r_ecef_from_llh, r_ecef_from_lla)))
print("ecef2geodetic split-array vs Nx3 lat allclose:", bool(np.allclose(lat_xyz, lat_ecef)))
print("ecef2geodetic split-array max |lat err| [rad]:", float(np.max(np.abs(lat_xyz - lat_vec))))
print("ecef2geodetic split-array max |lon err| [rad]:", float(np.max(np.abs(wrap_pi(lon_xyz - lon_vec)))))
print("ecef2geodetic split-array max |h err| [m]:", float(np.max(np.abs(h_xyz - h_vec))))
print("first row ecef2geodetic(..., degrees=True) [deg,deg,m]:", float(lat_deg[0]), float(lon_deg[0]), float(h_deg[0]))


geodetic2ecef split-array vs Nx3 allclose: True
ecef2geodetic split-array vs Nx3 lat allclose: True
ecef2geodetic split-array max |lat err| [rad]: 9.992007221626409e-15
ecef2geodetic split-array max |lon err| [rad]: 0.0
ecef2geodetic split-array max |h err| [m]: 4.284083843231201e-08
first row ecef2geodetic(..., degrees=True) [deg,deg,m]: 34.05000000000003 -117.85 700.0000000018626


## 3) ENU Basis, Scalar ENU/ECEF/Geodetic, And Delta Rotations

This section uses:
- `enu_basis_from_latlon`, `enu_basis_from_ecef_xyz`
- `geodetic2enu`, `ecef2enu`, `enu2ecef`, `enu2geodetic`
- `ecef2enu_delta`, `enu2ecef_delta`


In [7]:
# Pick first target as scalar example.
lat_t = float(lat_vec[0])
lon_t = float(lon_vec[0])
h_t = float(h_vec[0])

e_m, n_m, u_m = tf.geodetic2enu(lat_t, lon_t, h_t, obs_lat, obs_lon, obs_h_m)
x_t, y_t, z_t = tf.enu2ecef(e_m, n_m, u_m, obs_lat, obs_lon, obs_h_m)
e_chk, n_chk, u_chk = tf.ecef2enu(x_t, y_t, z_t, obs_lat, obs_lon, obs_h_m)

lat_rt, lon_rt, h_rt = tf.enu2geodetic(e_m, n_m, u_m, obs_lat, obs_lon, obs_h_m)

dx_m, dy_m, dz_m = tf.enu2ecef_delta(e_m, n_m, u_m, obs_lat, obs_lon)
e_d, n_d, u_d = tf.ecef2enu_delta(dx_m, dy_m, dz_m, obs_lat, obs_lon)

basis_ll = tf.enu_basis_from_latlon(obs_lat, obs_lon)
x_obs, y_obs, z_obs = tf.geodetic2ecef(obs_lat, obs_lon, obs_h_m)
basis_xyz = tf.enu_basis_from_ecef_xyz(x_obs, y_obs, z_obs)

print("ENU [m]:", e_m, n_m, u_m)
print("ECEF->ENU backcheck max abs diff [m]:", float(np.max(np.abs([e_chk - e_m, n_chk - n_m, u_chk - u_m]))))
print("ENU->geodetic max errors [rad,rad,m]:", float(abs(lat_rt - lat_t)), float(abs(wrap_pi(lon_rt - lon_t))), float(abs(h_rt - h_t)))
print("delta roundtrip max abs diff [m]:", float(np.max(np.abs([e_d - e_m, n_d - n_m, u_d - u_m]))))
print("basis(latlon) orthonormal check ||R^T R - I||:", float(np.linalg.norm(basis_ll.T @ basis_ll - np.eye(3))))
print("basis(ecef xyz) orthonormal check ||R^T R - I||:", float(np.linalg.norm(basis_xyz.T @ basis_xyz - np.eye(3))))


ENU [m]: 13851.095888197668 5556.890799864916 432.54844567977443
ECEF->ENU backcheck max abs diff [m]: 0.0
ENU->geodetic max errors [rad,rad,m]: 5.551115123125783e-16 0.0 1.862645149230957e-09
delta roundtrip max abs diff [m]: 9.094947017729282e-13
basis(latlon) orthonormal check ||R^T R - I||: 3.215840282420699e-16
basis(ecef xyz) orthonormal check ||R^T R - I||: 1.0728407053915191e-16


## 4) ENU Unified Array APIs

Demonstrates unified ENU array inputs:
- split arrays such as `geodetic2enu(lat_vec, lon_vec, h_vec, ...)`
- `(N, 3)` arrays such as `geodetic2enu(lla_rad_m, lat0_rad=..., lon0_rad=..., h0_m=...)`
- the same input options for `ecef2enu`, `enu2ecef`, and `enu2geodetic`


In [8]:
e_llh, n_llh, u_llh = tf.geodetic2enu(lat_vec, lon_vec, h_vec, obs_lat, obs_lon, obs_h_m)
enu_stack = np.column_stack((e_llh, n_llh, u_llh))

enu_from_lla = np.column_stack(tf.geodetic2enu(np.column_stack((lat_vec, lon_vec, h_vec)), lat0_rad=obs_lat, lon0_rad=obs_lon, h0_m=obs_h_m))

x_from_enu, y_from_enu, z_from_enu = tf.enu2ecef(e_llh, n_llh, u_llh, obs_lat, obs_lon, obs_h_m)
r_ecef_from_enu = np.column_stack((x_from_enu, y_from_enu, z_from_enu))
r_ecef_from_enu3 = np.column_stack(tf.enu2ecef(enu_stack, lat0_rad=obs_lat, lon0_rad=obs_lon, h0_m=obs_h_m))

e_xyz, n_xyz, u_xyz = tf.ecef2enu(x_from_enu, y_from_enu, z_from_enu, obs_lat, obs_lon, obs_h_m)
e_ecef, n_ecef, u_ecef = tf.ecef2enu(r_ecef_from_enu, lat0_rad=obs_lat, lon0_rad=obs_lon, h0_m=obs_h_m)

lat_enu, lon_enu, h_enu = tf.enu2geodetic(e_llh, n_llh, u_llh, obs_lat, obs_lon, obs_h_m)
lat_enu3, lon_enu3, h_enu3 = tf.enu2geodetic(enu_stack, lat0_rad=obs_lat, lon0_rad=obs_lon, h0_m=obs_h_m)

print("geodetic2enu split-array vs Nx3 allclose:", bool(np.allclose(enu_stack, enu_from_lla)))
print("enu2ecef split-array vs Nx3 allclose:", bool(np.allclose(r_ecef_from_enu, r_ecef_from_enu3)))
print("ecef2enu split-array vs Nx3 allclose:", bool(np.allclose(np.column_stack((e_xyz, n_xyz, u_xyz)), np.column_stack((e_ecef, n_ecef, u_ecef)))))
print("enu2geodetic split-array vs Nx3 max |h diff| [m]:", float(np.max(np.abs(h_enu - h_enu3))))
print("enu2geodetic max |lat err| [rad]:", float(np.max(np.abs(lat_enu - lat_vec))))
print("enu2geodetic max |lon err| [rad]:", float(np.max(np.abs(wrap_pi(lon_enu - lon_vec)))))


geodetic2enu split-array vs Nx3 allclose: True
enu2ecef split-array vs Nx3 allclose: True
ecef2enu split-array vs Nx3 allclose: True
enu2geodetic split-array vs Nx3 max |h diff| [m]: 0.0
enu2geodetic max |lat err| [rad]: 9.992007221626409e-15
enu2geodetic max |lon err| [rad]: 0.0


## 5) AER Scalar Workflows

Demonstrates scalar AER path functions:
- `enu2aer`, `geodetic2aer`, `ecef2aer`
- `aer2enu`, `aer2ecef`, `aer2geodetic`


In [9]:
az_g, el_g, sr_g = tf.geodetic2aer(lat_t, lon_t, h_t, obs_lat, obs_lon, obs_h_m)
az_e, el_e, sr_e = tf.ecef2aer(x_t, y_t, z_t, obs_lat, obs_lon, obs_h_m)
az_enu, el_enu, sr_enu = tf.enu2aer(e_m, n_m, u_m)

e_from_aer, n_from_aer, u_from_aer = tf.aer2enu(az_g, el_g, sr_g)
x_from_aer, y_from_aer, z_from_aer = tf.aer2ecef(az_g, el_g, sr_g, obs_lat, obs_lon, obs_h_m)
lat_from_aer, lon_from_aer, h_from_aer = tf.aer2geodetic(az_g, el_g, sr_g, obs_lat, obs_lon, obs_h_m)

print("geodetic2aer vs ecef2aer [az,el,sr] close:", bool(np.allclose([az_g, el_g, sr_g], [az_e, el_e, sr_e])))
print("geodetic2aer vs enu2aer [az,el,sr] close:", bool(np.allclose([az_g, el_g, sr_g], [az_enu, el_enu, sr_enu])))
print("aer2enu backcheck max abs diff [m]:", float(np.max(np.abs([e_from_aer - e_m, n_from_aer - n_m, u_from_aer - u_m]))))
print("aer2ecef backcheck max abs diff [m]:", float(np.max(np.abs([x_from_aer - x_t, y_from_aer - y_t, z_from_aer - z_t]))))
print("aer2geodetic max errors [rad,rad,m]:", float(abs(lat_from_aer - lat_t)), float(abs(wrap_pi(lon_from_aer - lon_t))), float(abs(h_from_aer - h_t)))


geodetic2aer vs ecef2aer [az,el,sr] close: True
geodetic2aer vs enu2aer [az,el,sr] close: True
aer2enu backcheck max abs diff [m]: 1.8189894035458565e-12
aer2ecef backcheck max abs diff [m]: 0.0
aer2geodetic max errors [rad,rad,m]: 5.551115123125783e-16 0.0 1.862645149230957e-09


## 6) AER Unified Array APIs

Demonstrates the unified AER array interface:
- split arrays such as `geodetic2aer(lat_vec, lon_vec, h_vec, ...)`
- `(N, 3)` arrays such as `aer2ecef(aer_rad_m, lat0_rad=..., lon0_rad=..., h0_m=...)`
- the same input options for `ecef2aer` and `aer2geodetic`


In [10]:
az_vec, el_vec, sr_vec = tf.geodetic2aer(lat_vec, lon_vec, h_vec, obs_lat, obs_lon, obs_h_m)

x_a, y_a, z_a = tf.aer2ecef(az_vec, el_vec, sr_vec, obs_lat, obs_lon, obs_h_m)
r_ecef_aer = np.column_stack((x_a, y_a, z_a))
r_ecef_aer3 = np.column_stack(tf.aer2ecef(np.column_stack((az_vec, el_vec, sr_vec)), lat0_rad=obs_lat, lon0_rad=obs_lon, h0_m=obs_h_m))

az_xyz, el_xyz, sr_xyz = tf.ecef2aer(x_a, y_a, z_a, obs_lat, obs_lon, obs_h_m)

lat_a, lon_a, h_a = tf.aer2geodetic(az_vec, el_vec, sr_vec, obs_lat, obs_lon, obs_h_m)
lat_a3, lon_a3, h_a3 = tf.aer2geodetic(np.column_stack((az_vec, el_vec, sr_vec)), lat0_rad=obs_lat, lon0_rad=obs_lon, h0_m=obs_h_m)

print("aer2ecef split-array vs Nx3 allclose:", bool(np.allclose(r_ecef_aer, r_ecef_aer3)))
print("geodetic2aer split-array vs ecef2aer split-array allclose:", bool(np.allclose(np.column_stack((az_vec, el_vec, sr_vec)), np.column_stack((az_xyz, el_xyz, sr_xyz)))))
print("aer2geodetic split-array vs Nx3 max |h diff| [m]:", float(np.max(np.abs(h_a - h_a3))))
print("aer2geodetic max |lat err| [rad]:", float(np.max(np.abs(lat_a - lat_vec))))
print("aer2geodetic max |lon err| [rad]:", float(np.max(np.abs(wrap_pi(lon_a - lon_vec)))))


aer2ecef split-array vs Nx3 allclose: True
geodetic2aer split-array vs ecef2aer split-array allclose: True
aer2geodetic split-array vs Nx3 max |h diff| [m]: 0.0
aer2geodetic max |lat err| [rad]: 9.992007221626409e-15
aer2geodetic max |lon err| [rad]: 0.0


## 7) Coarse ECI <-> ECEF And Coarse ECI -> Geodetic

This section exercises the coarse-transform functions:
- scalar position and position/velocity variants
- vectorized position and position/velocity variants
- geodetic outputs in radians and degrees via `degrees=True`


In [11]:
epoch = Time("2026-01-01T00:00:00", scale="utc")
dt_s = np.arange(0.0, 600.0, 60.0, dtype=np.float64)
times = epoch + dt_s * u.s

radius_m = 7000e3
omega = 2.0 * np.pi / (95.0 * 60.0)
theta = omega * dt_s

r_eci = np.column_stack(
    (
        radius_m * np.cos(theta),
        radius_m * np.sin(theta),
        1.0e5 * np.sin(0.5 * theta),
    )
).astype(np.float64)

v_eci = np.column_stack(
    (
        -radius_m * omega * np.sin(theta),
        radius_m * omega * np.cos(theta),
        1.0e5 * 0.5 * omega * np.cos(0.5 * theta),
    )
).astype(np.float64)

jd_ut1 = times.ut1.jd.astype(np.float64)
jd_tt = times.tt.jd.astype(np.float64)

# Scalar calls
x_ecef0, y_ecef0, z_ecef0 = tf.coarse_eci2ecef_pos(*r_eci[0], jd_ut1[0], jd_tt[0])
x_eci0, y_eci0, z_eci0 = tf.coarse_ecef2eci_pos(x_ecef0, y_ecef0, z_ecef0, jd_ut1[0], jd_tt[0])

pv_ecef0 = tf.coarse_eci2ecef_pos_vel(*r_eci[0], *v_eci[0], jd_ut1[0], jd_tt[0])
pv_eci0 = tf.coarse_ecef2eci_pos_vel(*pv_ecef0, jd_ut1[0], jd_tt[0])

lat0_rad, lon0_rad, h0_m = tf.coarse_eci2geodetic(*r_eci[0], jd_ut1[0], jd_tt[0])
lat0_deg, lon0_deg, h0_deg_m = tf.coarse_eci2geodetic(*r_eci[0], jd_ut1[0], jd_tt[0], degrees=True)

# Batch calls with stacked (N, 3) arrays
coarse_pos = tf.coarse_eci2ecef_pos(r_eci, jd_ut1=jd_ut1, jd_tt=jd_tt)
r_ecef = np.column_stack(coarse_pos)
r_eci_back = np.column_stack(tf.coarse_ecef2eci_pos(r_ecef, jd_ut1=jd_ut1, jd_tt=jd_tt))

coarse_state = tf.coarse_eci2ecef_pos_vel(r_eci, v_eci, jd_ut1=jd_ut1, jd_tt=jd_tt)
r_ecef_pv = np.column_stack(coarse_state[:3])
v_ecef_pv = np.column_stack(coarse_state[3:])
back_state = tf.coarse_ecef2eci_pos_vel(r_ecef_pv, v_ecef_pv, jd_ut1=jd_ut1, jd_tt=jd_tt)
r_eci_pv_back = np.column_stack(back_state[:3])
v_eci_pv_back = np.column_stack(back_state[3:])

lat_rad_vec, lon_rad_vec, h_m_vec = tf.coarse_eci2geodetic(r_eci, jd_ut1=jd_ut1, jd_tt=jd_tt)
lat_deg_vec, lon_deg_vec, h_deg_vec = tf.coarse_eci2geodetic(r_eci, jd_ut1=jd_ut1, jd_tt=jd_tt, degrees=True)

err_pos = np.linalg.norm(r_eci_back - r_eci, axis=1)
err_pos_pv = np.linalg.norm(r_eci_pv_back - r_eci, axis=1)
err_vel_pv = np.linalg.norm(v_eci_pv_back - v_eci, axis=1)

print("Scalar position roundtrip exact match:", bool(np.allclose([x_eci0, y_eci0, z_eci0], r_eci[0])))
print("Scalar state roundtrip exact match:", bool(np.allclose(pv_eci0, np.concatenate((r_eci[0], v_eci[0])))))
print("Batch position roundtrip max error [m]:", float(np.max(err_pos)))
print("Batch pos+vel roundtrip max position error [m]:", float(np.max(err_pos_pv)))
print("Batch pos+vel roundtrip max velocity error [m/s]:", float(np.max(err_vel_pv)))
print("First coarse geodetic sample [rad,rad,m]:", float(lat0_rad), float(lon0_rad), float(h0_m))
print("First coarse geodetic sample [deg,deg,m]:", float(lat0_deg), float(lon0_deg), float(h0_deg_m))
print("coarse_eci2geodetic first row [deg,deg,m]:", float(lat_deg_vec[0]), float(lon_deg_vec[0]), float(h_deg_vec[0]))



Scalar position roundtrip exact match: True
Scalar state roundtrip exact match: True
Batch position roundtrip max error [m]: 2.8325163752310304e-09
Batch pos+vel roundtrip max position error [m]: 2.8325163752310304e-09
Batch pos+vel roundtrip max velocity error [m/s]: 3.641647632000604e-12
First coarse geodetic sample [rad,rad,m]: 0.0025516362151520693 -1.7510543899275524 621863.1381508755
First coarse geodetic sample [deg,deg,m]: 0.14619798598094885 -100.32802624070392 621863.1381508755
coarse_eci2geodetic first row [deg,deg,m]: 0.14619798598094885 -100.32802624070392 621863.1381508755


## 8) Calling Unified Transforms From Your Own `@numba.njit` Function

This is the pattern you would use in your own code:
- write a normal `@njit` function
- call the public `nstk.transforms` functions directly inside it
- use the same unified scalar / split-array / `(N, 3)` signatures shown elsewhere in this notebook
- keep using keyword arguments for matrix-form calls that need extra parameters
- in notebooks, prefer plain `@njit`; add `cache=True` later if you move the helper into a normal `.py` module

`transform(...)` is not included here because it is the Orekit-backed runtime path, not a Numba API.


In [12]:
@njit
def observer_products_jit(lla_rad_m, lat0_rad, lon0_rad, h0_m):
    x_m, y_m, z_m = tf.geodetic2ecef(lla_rad_m)
    e_m, n_m, u_m = tf.ecef2enu(x_m, y_m, z_m, lat0_rad, lon0_rad, h0_m)
    az_rad, el_rad, srange_m = tf.ecef2aer(x_m, y_m, z_m, lat0_rad, lon0_rad, h0_m)
    return e_m, n_m, u_m, az_rad, el_rad, srange_m

@njit
def coarse_products_jit(r_eci_m, jd_ut1_arr, jd_tt_arr):
    x_ecef_m, y_ecef_m, z_ecef_m = tf.coarse_eci2ecef_pos(
        r_eci_m,
        jd_ut1=jd_ut1_arr,
        jd_tt=jd_tt_arr,
    )
    lat_deg, lon_deg, h_m = tf.coarse_eci2geodetic(
        r_eci_m,
        jd_ut1=jd_ut1_arr,
        jd_tt=jd_tt_arr,
        degrees=True,
    )
    return x_ecef_m, y_ecef_m, z_ecef_m, lat_deg, lon_deg, h_m

jit_local = observer_products_jit(np.column_stack((lat_vec, lon_vec, h_vec)), obs_lat, obs_lon, obs_h_m)
jit_coarse = coarse_products_jit(r_eci, jd_ut1, jd_tt)

print(
    "observer_products_jit first row [E,N,U,az_deg,el_deg,range_m]:",
    np.array(
        [
            jit_local[0][0],
            jit_local[1][0],
            jit_local[2][0],
            np.rad2deg(jit_local[3][0]),
            np.rad2deg(jit_local[4][0]),
            jit_local[5][0],
        ]
    ),
)
print(
    "coarse_products_jit first row [x,y,z,lat_deg,lon_deg,h_m]:",
    np.array(
        [
            jit_coarse[0][0],
            jit_coarse[1][0],
            jit_coarse[2][0],
            jit_coarse[3][0],
            jit_coarse[4][0],
            jit_coarse[5][0],
        ]
    ),
)


observer_products_jit first row [E,N,U,az_deg,el_deg,range_m]: [13851.095888  5556.8908     432.548446    68.139945     1.66014
 14930.471889]
coarse_products_jit first row [x,y,z,lat_deg,lon_deg,h_m]: [-1254980.192914 -6886560.067598    17752.486019        0.146198
     -100.328026   621863.138151]


## 9) Timed Orekit-Backed Frame Transform (`transform`)

`transform(...)` handles arbitrary frame transforms with time dependence.

Notes:
- `time` accepts `astropy.time.Time`, Orekit `AbsoluteDate`, unix-second numerics, or time `Quantity`.
- For numeric inputs, values are interpreted as **unix seconds**.
- If you pass `Quantity` vectors in, outputs preserve quantity units.
- Frame strings can be canonical names (for example `"gcrf"`, `"itrf"`, `"teme"`), aliases (`"eci"`, `"ecef"`, `"j2000"`), versioned names (`"itrf2014"`, `"tod2010"`), or full Orekit `Predefined` enum names.
- For anything more specialized, you can pass an Orekit `Frame` object directly.
- This section includes a direct arbitrary-frame example (`"teme" -> "mod"`) in addition to Earth-fixed transforms.


In [ ]:

times_tf = epoch + np.array([0.0, 45.0, 90.0], dtype=np.float64) * u.s

r_gcrf = r_eci[:3].copy()
v_gcrf = v_eci[:3].copy()
a_gcrf = np.zeros_like(r_gcrf)

# Position-only transform.
p_itrf_only, _, _ = tf.transform(
    from_frame="gcrf",
    to_frame="itrf",
    time=times_tf,
    position=r_gcrf,
)

# Position + velocity transform and roundtrip.
p_itrf, v_itrf, _ = tf.transform(
    from_frame="gcrf",
    to_frame="itrf",
    time=times_tf,
    position=r_gcrf,
    velocity=v_gcrf,
)
p_back, v_back, _ = tf.transform("itrf", "gcrf", times_tf, p_itrf, velocity=v_itrf)

# Arbitrary timed frame pair: TEME -> MOD.
r_teme = np.array(
    [
        [7000e3, 0.0, 0.0],
        [6995e3, 80e3, 10e3],
        [6980e3, 160e3, 20e3],
    ],
    dtype=np.float64,
)
r_mod, _, _ = tf.transform("teme", "mod", times_tf, r_teme)

# Alias frames ("eci"->"gcrf", "ecef"->"itrf").
p_alias, _, _ = tf.transform("eci", "ecef", times_tf, r_gcrf)

# Quantity-preserving path with acceleration.
p_q, v_q, a_q = tf.transform(
    from_frame="gcrf",
    to_frame="itrf",
    time=times_tf,
    position=r_gcrf * u.m,
    velocity=v_gcrf * (u.m / u.s),
    acceleration=a_gcrf * (u.m / (u.s**2)),
)

# Acceleration-only mode: velocity output stays None.
_, v_none, a_only = tf.transform(
    from_frame="gcrf",
    to_frame="itrf",
    time=times_tf,
    position=r_gcrf,
    acceleration=a_gcrf,
)

# Equivalent time forms: astropy Time, unix seconds, Orekit AbsoluteDate list.
unix_times = times_tf.utc.unix.astype(np.float64)
p_unix, _, _ = tf.transform("gcrf", "itrf", unix_times, r_gcrf)

t0_abs = astropy_time_to_orekit_date(times_tf[0])
abs_times = [t0_abs.shiftedBy(0.0), t0_abs.shiftedBy(45.0), t0_abs.shiftedBy(90.0)]
p_abs, _, _ = tf.transform("gcrf", "itrf", abs_times, r_gcrf)

print("position-only output shape:", p_itrf_only.shape)
print("PV output shapes:", p_itrf.shape, v_itrf.shape)
print("teme->mod output shape:", r_mod.shape, "first sample [m]:", r_mod[0])
print("gcrf->itrf->gcrf max position error [m]:", float(np.max(np.linalg.norm(p_back - r_gcrf, axis=1))))
print("gcrf->itrf->gcrf max velocity error [m/s]:", float(np.max(np.linalg.norm(v_back - v_gcrf, axis=1))))
print("alias frames match canonical names:", bool(np.allclose(p_alias, p_itrf_only)))
print("Quantity output types:", type(p_q).__name__, type(v_q).__name__, type(a_q).__name__)
print("acceleration-only returns velocity None:", v_none is None)
print("time forms consistent (Time vs unix):", bool(np.allclose(p_itrf_only, p_unix)))
print("time forms consistent (Time vs AbsoluteDate):", bool(np.allclose(p_itrf_only, p_abs)))


### Discovering Supported Frame Names

The transform API accepts a mix of NSTK-friendly aliases and Orekit frame names. The cell below shows a practical way to inspect what is available in the current runtime.


In [14]:
# Orekit enums become importable once the runtime is initialized.
from nstk._orekit_runtime import ensure_orekit_runtime

ensure_orekit_runtime()

from org.orekit.frames import ITRFVersion, Predefined
from org.orekit.utils import IERSConventions

common_named_frames = [
    "gcrf",
    "icrf",
    "eme2000",
    "teme",
    "mod",
    "tod",
    "cirf",
    "gtod",
    "tirf",
    "ecliptic",
    "itrf",
    "itrfcio",
    "itrfequinox",
    "veis1950",
]
frame_aliases = {
    "eci": "gcrf",
    "ecef": "itrf",
    "itrs": "itrf",
    "j2000": "eme2000",
    "gcrs": "gcrf",
    "meanofdate": "mod",
    "trueofdate": "tod",
}
itrf_versions = sorted(
    (
        attr.removeprefix("ITRF_")
        for attr in dir(ITRFVersion)
        if attr.startswith("ITRF_") and attr.removeprefix("ITRF_").isdigit()
    ),
    key=int,
)
iers_conventions = sorted(
    (
        attr.removeprefix("IERS_")
        for attr in dir(IERSConventions)
        if attr.startswith("IERS_") and attr.removeprefix("IERS_").isdigit()
    ),
    key=int,
)
predefined_frame_names = sorted(name for name in dir(Predefined) if name.isupper())

latest_itrf_name = f"itrf{itrf_versions[-1]}"
latest_tod_name = f"tod{iers_conventions[-1]}"

print("Common named frames:", common_named_frames)
print("Alias shortcuts:", frame_aliases)
print("Available ITRF versions:", itrf_versions)
print("Available IERS conventions:", iers_conventions)
print("Example versioned strings:", [latest_itrf_name, latest_tod_name, f"mod{iers_conventions[0]}"])
print("First 12 Predefined names:", predefined_frame_names[:12])
print("Total Predefined names:", len(predefined_frame_names))


Common named frames: ['gcrf', 'icrf', 'eme2000', 'teme', 'mod', 'tod', 'cirf', 'gtod', 'tirf', 'ecliptic', 'itrf', 'itrfcio', 'itrfequinox', 'veis1950']
Alias shortcuts: {'eci': 'gcrf', 'ecef': 'itrf', 'itrs': 'itrf', 'j2000': 'eme2000', 'gcrs': 'gcrf', 'meanofdate': 'mod', 'trueofdate': 'tod'}
Available ITRF versions: ['1988', '1989', '1990', '1991', '1992', '1993', '1994', '1996', '1997', '2000', '2005', '2008', '2014', '2020']
Available IERS conventions: ['1996', '2003', '2010']
Example versioned strings: ['itrf2020', 'tod2010', 'mod1996']
First 12 Predefined names: ['CIRF_CONVENTIONS_1996_ACCURATE_EOP', 'CIRF_CONVENTIONS_1996_SIMPLE_EOP', 'CIRF_CONVENTIONS_2003_ACCURATE_EOP', 'CIRF_CONVENTIONS_2003_SIMPLE_EOP', 'CIRF_CONVENTIONS_2010_ACCURATE_EOP', 'CIRF_CONVENTIONS_2010_SIMPLE_EOP', 'ECLIPTIC_CONVENTIONS_1996', 'ECLIPTIC_CONVENTIONS_2003', 'ECLIPTIC_CONVENTIONS_2010', 'EME2000', 'GCRF', 'GTOD_CONVENTIONS_1996_ACCURATE_EOP']
Total Predefined names: 51


### More `transform(...)` Patterns

These extra examples show version-pinned frame strings, a direct Orekit `Frame` object, and the broadcasting behavior when one state is paired with many times or vice versa.


In [15]:
from org.orekit.frames import FramesFactory

# Pin the exact frame/convention through versioned string forms.
p_itrf_latest_named, _, _ = tf.transform("gcrf", latest_itrf_name, times_tf, r_gcrf)
p_tod_default, _, _ = tf.transform("gcrf", "tod", times_tf, r_gcrf)
p_tod_latest_named, _, _ = tf.transform("gcrf", latest_tod_name, times_tf, r_gcrf)

# Full Orekit Predefined enum names are accepted too.
predefined_name = next(
    (name for name in predefined_frame_names if name.startswith("TOD_CONVENTIONS_2010")),
    predefined_frame_names[0],
)
p_predefined, _, _ = tf.transform("gcrf", predefined_name, times_tf, r_gcrf)

# Passing an Orekit Frame object directly is equivalent.
orekit_frame = FramesFactory.getFrame(getattr(Predefined, predefined_name))
p_frame_obj, _, _ = tf.transform("gcrf", orekit_frame, times_tf, r_gcrf)

# Broadcasting examples.
p_single_state_history, _, _ = tf.transform("gcrf", "itrf", times_tf, r_gcrf[0])
p_batch_same_epoch, _, _ = tf.transform("gcrf", "itrf", times_tf[0], r_gcrf)

print("latest versioned ITRF matches unversioned 'itrf':", bool(np.allclose(p_itrf_latest_named, p_itrf_only)))
print("latest versioned TOD matches unversioned 'tod':", bool(np.allclose(p_tod_latest_named, p_tod_default)))
print("Predefined name used:", predefined_name)
print("Predefined name matches direct Orekit Frame:", bool(np.allclose(p_predefined, p_frame_obj)))
print("single state broadcast across times shape:", p_single_state_history.shape)
print("multiple states at one epoch shape:", p_batch_same_epoch.shape)
print("Inspect `predefined_frame_names` for the full Predefined list.")


latest versioned ITRF matches unversioned 'itrf': True
latest versioned TOD matches unversioned 'tod': True
Predefined name used: TOD_CONVENTIONS_2010_ACCURATE_EOP
Predefined name matches direct Orekit Frame: True
single state broadcast across times shape: (3, 3)
multiple states at one epoch shape: (3, 3)
Inspect `predefined_frame_names` for the full Predefined list.


## 9) Constants Cheat Sheet


In [16]:
print("WGS84_A [m]:", tf.WGS84_A)
print("WGS84_B [m]:", tf.WGS84_B)
print("WGS84_E2:", tf.WGS84_E2)
print("DEG2RAD / RAD2DEG:", tf.DEG2RAD, tf.RAD2DEG)
print("J2000_JD:", tf.J2000_JD)
print("EARTH_OMEGA [rad/s]:", tf.EARTH_OMEGA)
print("DAS2R [rad/arcsec]:", tf.DAS2R)


WGS84_A [m]: 6378137.0
WGS84_B [m]: 6356752.314245179
WGS84_E2: 0.00669437999014133
DEG2RAD / RAD2DEG: 0.017453292519943295 57.29577951308232
J2000_JD: 2451545.0
EARTH_OMEGA [rad/s]: 7.292115e-05
DAS2R [rad/arcsec]: 4.84813681109536e-06


Next notebook: **03 - Walker Constellation**.